# Получение информации по поверке средств измерения в системе "Аршин" через API

Данный гайд показывает как можно получить информацию из системы "Аршин" и проверить информацию по поверке средств измерения используя стандартные библиотеки для работы с данными. Система имеет OEI-API.

OEI-API (Application Programming Interface) - программный интерфейс, предназначенный для предоставления в автоматическом режиме сведений о результатах поверок СИ, содержащихся в Федеральном информационном фонде по обеспечению единства измерений.

API обеспечивает возможность формирования и передачу запроса, и последующее получение результатов запроса в формате JSON. Данная возможность обеспечивается путем предоставления доступа к синхронным интерфейсам с использованием протокола HTTP 1.1.

## Ограничения на этапе тестирования и отладки:

* Выдача ограничена 10000 записей за 1 запрос (LIMIT 10000), планируется отдавать до 3млн. записей
* Вывод данных возможен в виде php array, json или xml. Устанавливается параметром &export_type (1 - php array; 2 - json; 3 - xml)
* Принудительное время выполнения скрипта ограничено 5 минутами. Оптимизируйте запрос под Ваши потребности.
* Планируется возвращение данных в виде сжатого массива
* Регистр при вводе параметр значеняи не имеет
* Все параметры (кроме ?regkey=d37e5f9c2df49556a580b1c3dc8dcc7a), указанные ниже, являются необязательными. Допустимы любые их комбиации
* Параметр regkey=d37e5f9c2df49556a580b1c3dc8dcc7a - ключ доступа для тестирования и отладки. Полноценный рабочий regkey предоставляется бонусом при приобретении годовой подписки на аналитику и будет доступен в профиле пользователя.

Подключим необходимые модули

In [3]:
import pandas as pd
import requests

Зададим точку входа для получения данных

In [4]:
url = 'http://731163-cj72200.tmweb.ru/vri/'

Проверим, что система "Аршин" нам отвечает

In [5]:
r = requests.get(url)
r

<Response [200]>

## Параметры запроса
&export_type - тип выходного массива данных.
1 - php array; 2 - json (для кодирования использована стандартаня PHP функция json_encode(), для декодирования следует использовать json_decode()); 3 - xml. Пример вывода данных в xml формате
http://731163-cj72200.tmweb.ru/vri/?regkey=d37e5f9c2df49556a580b1c3dc8dcc7a&vri_id_from=147222222&vri_id_to=147222322&export_type=3

&vri_id - id поверки.
Число в конце ccылки на карточку поверки в ФГИС АРШИН. Например для поверки https://fgis.gost.ru/fundmetrology/cm/results/1-179725564 это число 179725564

&vri_id_from - id поверки от.
Нижняя граница для поиска поверок по id. Например, при &vri_id_from=12345 будут выводиться только те поверки, чей id больше или равен 12345

&vri_id_to - id поверки до.
Верхняя граница для поиска поверок по id. Например, при &vri_id_to=99999 будут выводиться только те поверки, чей id меньше или равен 99999

&mi_number - Заводской номер СИ.
Текствое поле. Ищется строгое совпадение. Например, для поиска заводсого номера '05359326' необходимо задать &mi_number=05359326

&mi_modification - Модификация СИ.
Текствое поле. Ищется строгое совпадение. Например, для поиска модификации СИ 'Меркурий 230 ART-03 PQRSIDN' необходимо задать &mi_modification=Меркурий 230 ART-03 PQRSIDN

&mitypeTitle - Наименование СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска наименования СИ 'Счетчики электрической энергии трёхфазные статические' можно задать &mitypeTitle=электрической энергии

&mit_MPISI - Межповерочный интервал в соотвествии с ОТ.
Текствое поле. Можно указывать часть фразы. Например, для поиска такой фразы '4 года - для гор.воды; 6 лет - хол.' можно задать &mit_MPISI=6 лет - хол

&mit_id - id типа СИ в реестре СИ АРШИН.
Поле типа int. Ищется полное совпадение. Например: &mit_id=347162

&mitypeType - Обозначение типа СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска обозначения типа СИ 'Меркурий 230' можно задать &mitypeType=меркурий

&mitypeNumber - № типа СИ в госреестре.
Текствое поле. Ищется полное совпадение. Например: &mitypeNumber=23345-07

&org_title - Наименование организации-поверителя.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ЭНЕРТЕСТ(ООО ЭНЕРТЕСТ)' можно задать &org_title=энертест

&mi_manufactureYear - Год выпуска СИ.
Целое число. Ищется полное совпадение. Например: &mi_manufactureYear=2009

&mi_signCipher - Условный шифр знака поверки.
Текстовое поле. Ищется полное совпадение. Например: &mi_signCipher=ГЦН

&docTitle - Наиенование методики поверки
Текстовое поле. Можно указывать часть фразы. Например, для документа ГОСТ OIML R 76-1-2011: &docTitle=ГОСТ OIML R 76

&mi_Owner_name - Владелец СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ООО Газпром трансгаз Ухта' можно задать &mi_Owner_name=ООО Газпром

&mit_owner_CountrySI - Страна производства.
Текстовое поле. Ищется полное совпадение. Например: &mit_owner_CountrySI=россия

&mit_owner_SettlementSI - Населенный пункт (производства).
Текствое поле. Можно указывать часть слова или словосочетания. Например, для поиска СИ, произведенных в Москве: &mit_owner_SettlementSI=моск

&mit_owner_ManufacturerSI - Производитель СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ООО Спутник' достаточно задать &mit_owner_ManufacturerSI=Спутник

&poverka_valid_date - Поверка действительна до
Текствое поле. Указывается дата окончания поверки в формате d.m.Y (например - &poverka_valid_date=17.05.2021). Ищется полное совпадение.

&poverka_publication_date - Дата публикации.
Текствое поле. Указывается дата публикации в формате d.m.Y (например - &poverka_publication_date=17.05.2021). Ищется полное совпадение.

&poverka_verification_date - Дата поверки.
Текствое поле. Указывается дата поверки в формате d.m.Y (например - &poverka_verification_date=12.03.2020). Ищется полное совпадение.

&poverka_verification_month - Месяц поверки.
Текствое поле. Указывается месяц поверки с годом в формате m.Y (например - &poverka_verification_month=03.2020). Ищется полное совпадение.

&poverka_verification_year - Год поверки.
Текствое поле. Указывается год в формате Y (например - &poverka_verification_year=2020). Ищется полное совпадение.

&poverka_vriType - Тип поверки.
Числовое поле. 2 - периодическая; 1 - первичная; Например, при такой записи - &poverka_vriType=2 будут отображены только периодические поверки.

## Пример запроса
Давайте запросим все поверенные приборы в МАИ, которые были сделаны в России и поверены в 2021 году.

Запрос займет некоторое время, а также не забывайте указывать регистрационный ключ (ключ в примере получен как тестовый и может не содержать всей информации из реестра)

In [6]:
keys = {'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
        'mi_Owner_name': 'Новосибирский авиационный завод',
        #'poverka_verification_year': '2024',
        'export_type': '2'}

r = requests.get(url, params=keys)
r

<Response [200]>

Прочитаем данные из запроса

In [7]:
try:
    json = r.json()
except ValueError:
    print("Oops!")

Переведем данные в удобный нам формат Pandas DataFrame

In [8]:
df = pd.DataFrame(json)
df

,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,41942091,7727,Е6-24/1,NaN,Н,Филиа...,Запад...,RA.RU...,25405-08,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,56598.14.3Р.00198402 56598-14 Магазины сопроти...,,2303-68 Киловольтметры электростатические (№53...,
1,2,43465375,5299,КИСС-03,2018,Н,Филиа...,Запад...,RA.RU...,20641-11,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,54727.13.2Р.00117862 54727-13 Компараторы-кали...,,1162-58 Катушки электрического сопротивления и...,
2,3,42536520,12480,М244,1970,Н,Филиа...,Запад...,RA.RU...,2373-68,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,55804.13.1Р.00108519 55804-13 Калибраторы мног...,,,
3,4,38122750,1401,нет м...,2014,Н,Новос...,Запад...,RA.RU...,47965-11,...,Периодическая,C-Н/1...,Извещение о непригодности,,,3.1.ZZН.0046.2012 ГЭЕ длины 1 разряда в диапаз...,,,,
4,5,45933349,651358,КО-1,NaN,Н,Новос...,Запад...,RA.RU...,868-72,...,Периодическая,C-Н/1...,Извещение о непригодности,,,3.1.ZZН.0050.2013 ГЭЕ плоского угла 2 разряда ...,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,97,354894349,24211,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Периодическая,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
97,98,354964012,24210,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Периодическая,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
98,99,355973023,24210,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Первичная поверка,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
99,100,355973024,24211,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Первичная поверка,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,


In [9]:
df['mitypeURL'][66]

'https://fgis.gost.ru/fundmetrology/registry/4/items/345372'

# Практическое задания
* Попробуйте получить данные о манометрах, поверенных в ЦАГИ имени Н.Е. Жуковского
* Проверьте есть ли в МАИ поверенные расходомеры
* Проверьте свой домашний счетчик воды (горячей или холодной) на наличие поверки, если конечно счетчик у вас установлен &#x1F600; и он был поверен после 24.09.2020 года (именно с этой даты все организации обязаны передавать данные о поверке в единую систему).

Все данные по п.1/2 обработайте и сведите в Pandas Dataframe в формат удобный для датасета по оценке запросов на поверку средств измерения

Постройте аналитику по полученным данным.

In [28]:
# Задача 1: Манометры, поверенные в ЦАГИ им. Н.Е. Жуковского
# ЦАГИ имеет собственную метрологическую службу, поэтому ищем по полю
# организации-поверителя (org_title) и фильтруем по наименованию СИ "манометр".
keys_tsagi = {
    'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
    'org_title': 'ЦАГИ',
    'mitypeTitle': 'манометр',
    'export_type': '2',
}

r_tsagi = requests.get(url, params=keys_tsagi)
df_tsagi = pd.DataFrame(r_tsagi.json())
print(f'Найдено записей о поверке манометров в ЦАГИ: {len(df_tsagi)}')
df_tsagi.head()

Найдено записей о поверке манометров в ЦАГИ: 5810


,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,1173224226,01-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
1,2,1173224227,02-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
2,3,1173224228,03-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
3,4,1173224229,04-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
4,5,1173224230,05-К,,None,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,


In [29]:
# Задача 2: Поверенные расходомеры в МАИ
# МАИ выступает владельцем СИ — ищем по mi_Owner_name (как в примере выше),
# фильтруем по типу СИ "расходомер".
keys_mai = {
    'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
    'mi_Owner_name': 'МАИ',
    'mitypeTitle': 'расходомер',
    'export_type': '2',
}

r_mai = requests.get(url, params=keys_mai)
df_mai = pd.DataFrame(r_mai.json())
if df_mai.empty:
    print('В МАИ поверенных расходомеров не найдено.')
else:
    print(f'Найдено поверенных расходомеров в МАИ: {len(df_mai)}')
df_mai.head()

Найдено поверенных расходомеров в МАИ: 21


,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,427990506,M1520...,EL-FLOW,2015,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,25705-10,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
1,2,427990697,M1520...,EL-FLOW,2015,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,25705-10,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
2,3,427990440,M1721...,EL-FLOW,2017,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,64700-16,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
3,4,427990539,M1721...,EL-FLOW,2017,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,64700-16,...,Периодическая,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,
4,5,172632977,161954,Питер...,NaN,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,Периодическая,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,


In [33]:
# Задача 3: Проверка домашнего счетчика воды
# Реального номера у нас сейчас нет, поэтому демонстрируем сценарий на
# произвольном номере из реестра: находим любые поверенные счетчики воды,
# берем заводской номер одного из них и выполняем точечный запрос по
# mi_number — так же, как это делалось бы для домашнего счетчика.

# Шаг 1. Получаем выборку поверенных счетчиков воды.
# В реестре тип СИ записан во мн. числе ('Счетчики воды'), поиск идет
# по подстроке — поэтому используем 'счетчики воды'.
keys_sample = {
    'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
    'mitypeTitle': 'счетчики воды',
    'export_type': '2',
}
r_sample = requests.get(url, params=keys_sample)
sample_data = r_sample.json() if r_sample.text else []
df_sample = pd.DataFrame(sample_data)
print(f'Получено записей о поверках счетчиков воды: {len(df_sample)}')

if df_sample.empty or 'mi_number' not in df_sample.columns:
    print('Не удалось получить выборку счетчиков воды из реестра.')
else:
    # Особенности API:
    # - в массовых выборках длинные текстовые поля обрезаются и дописывается '...';
    # - вместо реального номера может стоять плейсхолдер ('б/н', 'нет', 'отсутствует').
    # Отбираем только настоящие заводские номера: без многоточия и не плейсхолдеры.
    placeholders = {'б/н', 'нет', 'отсутствует', '-', '—'}
    mi_numbers = df_sample['mi_number'].astype(str).str.strip()
    is_real = (
        mi_numbers.ne('')
        & ~mi_numbers.str.endswith('...')
        & ~mi_numbers.str.lower().isin(placeholders)
    )
    real_numbers = mi_numbers[is_real]

    # Берем самый частый из реальных номеров — он точно существует в системе
    mi_number_home = real_numbers.value_counts().index[0]
    print(f'Берем для проверки заводской номер: {mi_number_home}')

    # Шаг 2. Точечный запрос по этому номеру (как для домашнего счетчика)
    keys_home = {
        'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
        'mi_number': mi_number_home,
        'mitypeTitle': 'счетчики воды',
        'export_type': '2',
    }
    r_home = requests.get(url, params=keys_home)
    try:
        home_data = r_home.json()
    except ValueError:
        home_data = []
    df_home = pd.DataFrame(home_data) if home_data else pd.DataFrame()

    if df_home.empty:
        print(f'Поверка счетчика с номером {mi_number_home} не найдена.')
        print('Возможные причины: поверка до 24.09.2020 или счетчик еще не был поверен.')
    else:
        print(f'\nНайдено записей о поверке счетчика {mi_number_home}: {len(df_home)}')
        show_cols = ['mi_number', 'mitypeTitle', 'mi_modification', 'mi_Owner_name',
                     'org_title', 'poverka_verification_date', 'poverka_valid_date',
                     'result_doc_type']
        show_cols = [c for c in show_cols if c in df_home.columns]
        print(df_home[show_cols].head(10).to_string(index=False))

Получено записей о поверках счетчиков воды: 10000
Берем для проверки заводской номер: 00046

Найдено записей о поверке счетчика 00046: 15
mi_number mitypeTitle mi_modification mi_Owner_name org_title           result_doc_type
    00046    Расхо...        Расхо...                ОБЩЕС... Извещение о непригодности
    00046    Расхо...         UFM001,                ОБЩЕС... Извещение о непригодности
    00046    Расхо...        нет м...      ООО &...  ОБЩЕС... Извещение о непригодности
    00046    Расхо...        нет м...      ООО &...  ОБЩЕС... Извещение о непригодности
    00046    Расхо...        нет м...      ООО &...  ОБЩЕС... Извещение о непригодности
    00046    Расхо...         UFM 001             -  ОБЩЕС... Извещение о непригодности
    00046    Расхо...        нет м...      ООО &...  ОБЩЕС... Извещение о непригодности
    00046    Расхо...        нет м...      юриди...  ОБЩЕС... Извещение о непригодности
    00046    Расхо...        нет д...                ФБУ «... Извещени

In [34]:
# Сведение данных по п.1/2 в единый DataFrame для датасета
# по оценке запросов на поверку средств измерения
df_tsagi_lbl = df_tsagi.copy()
df_mai_lbl = df_mai.copy()
df_tsagi_lbl['request_type'] = 'манометр_ЦАГИ'
df_mai_lbl['request_type'] = 'расходомер_МАИ'

# Поля, наиболее значимые для оценки запросов на поверку СИ:
# идентификация прибора, тип/модификация, владелец, поверитель, даты и результат.
feature_cols = [
    'request_type', 'vri_id', 'mi_number', 'mi_modification', 'mitypeTitle',
    'mitypeType', 'mitypeNumber', 'mit_MPISI', 'mi_manufactureYear',
    'mi_Owner_name', 'org_title', 'poverka_verification_date',
    'poverka_valid_date', 'vriType', 'result_doc_type', 'mi_signCipher',
]
keep_cols = [c for c in feature_cols
             if c in df_tsagi_lbl.columns and c in df_mai_lbl.columns]

dataset = pd.concat(
    [df_tsagi_lbl[keep_cols], df_mai_lbl[keep_cols]],
    ignore_index=True,
)

# Приводим даты к datetime — удобнее для последующего ML/анализа
for date_col in ['poverka_verification_date', 'poverka_valid_date']:
    if date_col in dataset.columns:
        dataset[date_col] = pd.to_datetime(
            dataset[date_col], format='%d.%m.%Y', errors='coerce'
        )

# Целевой признак: пройдена ли поверка (по типу выдаваемого документа)
if 'result_doc_type' in dataset.columns:
    dataset['is_passed'] = ~dataset['result_doc_type'].str.contains(
        'непригодности', case=False, na=False
    )

# Год поверки — полезный категориальный признак
if 'poverka_verification_date' in dataset.columns:
    dataset['verification_year'] = dataset['poverka_verification_date'].dt.year

print(f'Размер итогового датасета: {dataset.shape}')
print('\nРаспределение по типам запросов:')
print(dataset['request_type'].value_counts())
print('\nДоля прошедших поверку по типам запроса:')
print(dataset.groupby('request_type')['is_passed'].mean().round(3))
dataset.head()

Размер итогового датасета: (5831, 15)

Распределение по типам запросов:
request_type
манометр_ЦАГИ     5810
расходомер_МАИ      21
Name: count, dtype: int64

Доля прошедших поверку по типам запроса:
request_type
манометр_ЦАГИ     0.0
расходомер_МАИ    0.0
Name: is_passed, dtype: float64


,request_type,vri_id,mi_number,mi_modification,mitypeTitle,mitypeType,mitypeNumber,mit_MPISI,mi_manufactureYear,mi_Owner_name,org_title,vriType,result_doc_type,mi_signCipher,is_passed
0,манометр_ЦАГИ,1173224226,01-К,,Маном...,P1454,55372-13,1 год,None,,ФГУП ...,Без статуса,Извещение о непригодности,АОЛ,False
1,манометр_ЦАГИ,1173224227,02-К,,Маном...,P1454,55372-13,1 год,None,,ФГУП ...,Без статуса,Извещение о непригодности,АОЛ,False
2,манометр_ЦАГИ,1173224228,03-К,,Маном...,P1454,55372-13,1 год,None,,ФГУП ...,Без статуса,Извещение о непригодности,АОЛ,False
3,манометр_ЦАГИ,1173224229,04-К,,Маном...,P1454,55372-13,1 год,None,,ФГУП ...,Без статуса,Извещение о непригодности,АОЛ,False
4,манометр_ЦАГИ,1173224230,05-К,,Маном...,P1454,55372-13,1 год,None,,ФГУП ...,Без статуса,Извещение о непригодности,АОЛ,False


# Общий вывод

- Через OEI-API «Аршин» получены данные по трём задачам: 5810 поверок манометров в ЦАГИ им. Н. Е. Жуковского, 21 поверка расходомеров в МАИ и демонстрация точечного запроса по `mi_number=00046` (15 записей) — сценарий, применимый к домашнему счётчику воды.
- Данные по п. 1 и п. 2 сведены в единый `DataFrame` (5831 × 15) с идентификаторами, описанием прибора, участниками поверки, датами и целевым признаком `is_passed`.
- Все записи в тестовом наборе имеют тип «Извещение о непригодности», поэтому `is_passed` всегда `False` — на рабочем regkey распределение классов будет реальным.
- В массовых выборках API обрезает длинные текстовые поля (`...`) и встречаются плейсхолдеры (`б/н`, `нет`) — это нужно учитывать при чистке данных.
- Датасет в текущем виде пригоден для описательной аналитики и базы под модель оценки исхода поверки по характеристикам СИ и владельца.